In [2]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import psutil
import torch.nn.functional as F
import random
import matplotlib.pyplot  as plt
%matplotlib inline

device = 'cuda'

g = torch.Generator(device=device).manual_seed(42) # for reproducability

In [3]:
class Linear:
    
    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out), generator=g, device=device)  *  (2 / fan_in)**0.5
        self.bias = torch.zeros(fan_out, device=device) if bias else None
        
        self.weight.requires_grad_(True)

        if self.bias is not None:
            self.bias.requires_grad_(True)
    
    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out
    
    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])

class BatchNorm1d:
    
    def __init__(self, dim, eps=1e-5, momentum=0.5):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        # params trained with backprop
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        # buffers trained with a running momentum
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)
        
    def __call__(self, x):
        if self.training:
            xmean = x.mean(0, keepdim=True)
            xvar = x.var(0, keepdim=True)
        else:
            xmean = self.running_mean
            xvar = self.running_var
        xhat = (x - xmean)/torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * self.running_mean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * self.running_var
            return self.out
        
    def parameters(self):
        return [self.gamma, self.beta]
    
class Tanh:
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out
    
    def parameters(self):
        return []
    
class ReLU:
    def __call__(self, x):
        self.out = torch.clamp(x, min=0)
        return self.out
    
    def parameters(self):
        return []
    
class Dropout:
    def __init__(self, p):
        if not 0 <= p < 1:
            raise ValueError("p must satisfy 0 <= p < 1")
        self.p = p
        self.training = True
    
    def __call__(self, x):
        if self.training == False:
            self.out = x
            return self.out
        
        random_values = torch.rand_like(x)
        mask = random_values >= self.p
        
        self.out = x*mask/(1-self.p)
        
        return self.out
    
    def parameters(self):
        return []

In [4]:
# fc1 = Linear(3072,100)
# fc2 = Linear(100, 10)
# batch_size = 64

# layers = [
#     Linear(3072, 100),
#     ReLU(),
#     Linear(100, 10)
# ]

In [5]:
images = torch.randn((32, 3, 32, 32), generator=g, device=device)
x = images.flatten(start_dim=1)

In [19]:
with torch.no_grad():
    layers[-1].weight /= 2**0.5 

In [ ]:
for layer in layers:
    x = layer(x)
    print(x.shape)


torch.Size([32, 100])
torch.Size([32, 100])
torch.Size([32, 10])


In [5]:
train_dataset = datasets.CIFAR10(
    root="../data_1",
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

In [6]:
image, label = train_dataset[0]

print(image.shape)
print(label)
print(train_dataset.classes[label])

torch.Size([3, 32, 32])
6
frog


In [7]:
split_generator = torch.Generator().manual_seed(42)

train_subset, val_subset = random_split(
    train_dataset,
    [45000, 5000],
    generator=split_generator
)

train_loader = DataLoader(
    train_subset, batch_size=64, shuffle=True
)

val_loader = DataLoader(
    val_subset, batch_size=64, shuffle=False
)

In [8]:
class Sequential:
    def __init__(self, layers):
        self.layers = layers
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def train(self):
        for layer in self.layers:
            if hasattr(layer, "training"):
                layer.training = True
    
    def eval(self):
        for layer in self.layers:
            if hasattr(layer, "training"):
                layer.training = False
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [9]:
g.manual_seed(2147483647)

layers = [
    Linear(3072, 100),
    ReLU(),
    Dropout(p=0.2),
    Linear(100, 10)
]

with torch.no_grad():
    layers[-1].weight /= 2**0.5

In [10]:
learning_rate = 0.001
model = Sequential(layers)

parameters = model.parameters()
velocities = [torch.zeros_like(p) for p in parameters]

first_moments = [torch.zeros_like(p) for p in parameters]
second_moments = [torch.zeros_like(p) for p in parameters]
t = 0

In [ ]:
## WITH MOMENTUM


# def train_one_epoch(model, loader, learning_rate, velocities, momentum):
    
#     for i, (images, labels) in enumerate(loader):
        
#         images = images.to(device)
#         labels = labels.to(device)
        
#         x = images.flatten(start_dim=1)
        
#         logits = model(x)
#         loss = F.cross_entropy(logits, labels)

#         for p in parameters:
#             p.grad = None
            
#         loss.backward()
    
#         with torch.no_grad():
#             for p, v in zip(parameters, velocities):
#                 v.mul_(momentum).add_(p.grad)
#                 p -= learning_rate * v
            
#         if i %100 == 0:
#             print(i, loss.item())

In [11]:
## WITH ADAM
def train_one_epoch(model, loader, learning_rate,first_moments, second_moments, t):
    
    model.train()
    parameters = model.parameters()
    
    for i, (images, labels) in enumerate(loader):
        
        images = images.to(device)
        labels = labels.to(device)
        
        x = images.flatten(start_dim=1)
        
        logits = model(x)
        loss = F.cross_entropy(logits, labels)

        for p in parameters:
            p.grad = None
            
        loss.backward()
    
        with torch.no_grad():
            t+=1
            for p, m ,s in zip(parameters, first_moments, second_moments):
                m.mul_(0.9).add_(0.1 * p.grad)
                s.mul_(0.999).add_(0.001 * p.grad**2)
                
                m_hat = m/(1-0.9**t)
                s_hat = s / (1 - 0.999**t)

                update = m_hat / (s_hat.sqrt() + 1e-8)
                p -= learning_rate * update
            
        # if i %100 == 0:
        #     print(i, loss.item())
    return t

In [13]:
for epoch in range(1, 8):
    t = train_one_epoch(
        model, 
        train_loader, 
        learning_rate, 
        # velocities, 
        # momentum=0.9
        first_moments,
        second_moments,
        t
    )
    print("Epoch:", epoch, "Updates:", t)
    print("Validation:", evaluate(model, val_loader))
    print("Training:", evaluate(model, train_loader))

Epoch: 1 Updates: 704
Validation: 33.74
Training: 33.33111111111111
Epoch: 2 Updates: 1408
Validation: 34.22
Training: 34.275555555555556
Epoch: 3 Updates: 2112
Validation: 36.86
Training: 37.05777777777778
Epoch: 4 Updates: 2816
Validation: 37.32
Training: 37.25333333333333
Epoch: 5 Updates: 3520
Validation: 38.32
Training: 38.76
Epoch: 6 Updates: 4224
Validation: 37.68
Training: 38.54666666666667
Epoch: 7 Updates: 4928
Validation: 39.18
Training: 38.87777777777778


In [12]:
def evaluate(model, loader):
    
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            x = images.flatten(start_dim=1)
            logits = model(x)

            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.shape[0]

    return 100 * correct / total

In [161]:
dropout = Dropout(p=0.2)
x = torch.ones(10000, device=device)

y = dropout(x)
print("Zero fraction:", (y == 0).float().mean().item())
print("Output mean:", y.mean().item())

dropout.training = False
print("Eval unchanged:", torch.equal(dropout(x), x))

Zero fraction: 0.19349999725818634
Output mean: 1.0081249475479126
Eval unchanged: True
